# Model Training

This notebook trains and compares Random Forest and XGBoost.
Saved model files are written to `../models/`.

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

In [2]:
MODELS_DIR = "../models"
PROCESSED_DIR = "../dataset/processed"

X_train = np.load(os.path.join(PROCESSED_DIR, "X_train.npy"), allow_pickle=True)
X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"), allow_pickle=True)
y_train = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"), allow_pickle=True)
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"), allow_pickle=True)

print(X_train.shape, X_test.shape)

(65865, 182) (16467, 182)


In [3]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.9444950507074755
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      7400
           1       0.97      0.93      0.95      9067

    accuracy                           0.94     16467
   macro avg       0.94      0.95      0.94     16467
weighted avg       0.95      0.94      0.94     16467

[[7120  280]
 [ 634 8433]]


In [4]:
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

XGBoost Accuracy: 0.9414586749256088
              precision    recall  f1-score   support

           0       0.91      0.96      0.94      7400
           1       0.97      0.92      0.95      9067

    accuracy                           0.94     16467
   macro avg       0.94      0.94      0.94     16467
weighted avg       0.94      0.94      0.94     16467

[[7139  261]
 [ 703 8364]]


In [5]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "Precision": [
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    "F1": [
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1
0,Random Forest,0.944495,0.967864,0.930076,0.948594
1,XGBoost,0.941459,0.969739,0.922466,0.945512


In [6]:
# Save both trained models in ../models/
joblib.dump(rf, os.path.join(MODELS_DIR, "random_forest_model.pkl"))
joblib.dump(xgb, os.path.join(MODELS_DIR, "xgboost_model.pkl"))

# Pick the higher-F1 model as the current best model.
if f1_score(y_test, y_pred_rf) >= f1_score(y_test, y_pred_xgb):
    best_model = rf
    best_model_name = "Random Forest"
else:
    best_model = xgb
    best_model_name = "XGBoost"

joblib.dump(best_model, os.path.join(MODELS_DIR, "best_model.pkl"))

print("Best model:", best_model_name)
print("Models saved in:", MODELS_DIR)

Best model: Random Forest
Models saved in: ../models


In [7]:
# Feature importance from Random Forest
preprocessor = joblib.load(os.path.join(MODELS_DIR, "preprocessor.pkl"))
feature_names = preprocessor.get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance["Feature"] = (
    importance["Feature"]
    .str.replace("remainder__", "", regex=False)
    .str.replace("encoder__", "", regex=False)
)

importance.head(15)

,Feature,Importance
157,sload,0.068112
172,smean,0.066980
154,sbytes,0.065972
156,rate,0.055473
162,dinpkt,0.053459
151,dur,0.049227
158,dload,0.047077
155,dbytes,0.045699
177,ct_dst_sport_ltm,0.042231
170,synack,0.041154
